In [2]:
import tkinter as tk
from tkinter import messagebox, simpledialog
import math
import time

# Глобальные переменные для хранения узлов и маршрутов
nodes = {}  # Словарь: имя узла -> (x, y)
edges = []  # Список: (начальный узел, конечный узел, расстояние)

# Функция для визуализации графа
def draw_graph(canvas):
    # Очищаем холст перед отрисовкой
    canvas.delete("all")
    
    # Рисуем все маршруты (ребра графа)
    for edge in edges:
        start, end, weight = edge
        x1, y1 = nodes[start]
        x2, y2 = nodes[end]
        
        canvas.create_line(x1, y1, x2, y2, fill="black", tags=f"edge_{start}_{end}")  # Линия между узлами
        canvas.create_text((x1 + x2) / 2, (y1 + y2) / 2, text=str(weight), tags=f"weight_{start}_{end}")  # Вес маршрута
    
    # Рисуем все узлы
    for node, (x, y) in nodes.items():
        radius = max(30, len(node) * 7)  # Размер узла зависит от длины имени
        canvas.create_oval(x - radius, y - radius, x + radius, y + radius, fill="white", outline="black", width=2, tags=node)  # Круг (узел)
        canvas.create_text(x, y, text=node, font=("Arial", 12, "bold"), fill="black", tags=f"text_{node}")  # Текст внутри узла

# Функция для поиска кратчайшего пути (Жадный алгоритм)
def greedy_algorithm():
    # Проверяем, есть ли узлы и маршруты
    if not nodes or not edges:
        return None, 0, []

    start_node = list(nodes.keys())[0]  # Начинаем с первого узла
    visited = [start_node]  # Список посещенных узлов
    total_distance = 0  # Суммарное расстояние
    route = []  # Маршрут

    # Пока не посетили все узлы
    while len(visited) < len(nodes):
        last_node = visited[-1]  # Последний посещенный узел
        min_distance = float('inf')  # Минимальное расстояние
        next_node = None  # Следующий узел

        # Ищем ближайший непосещенный узел
        for edge in edges:
            if edge[0] == last_node and edge[1] not in visited:
                if edge[2] < min_distance:
                    min_distance = edge[2]
                    next_node = edge[1]
            elif edge[1] == last_node and edge[0] not in visited:
                if edge[2] < min_distance:
                    min_distance = edge[2]
                    next_node = edge[0]

        # Если не нашли доступный узел, прерываем цикл
        if next_node is None:
            break

        # Добавляем узел в список посещенных
        visited.append(next_node)
        total_distance += min_distance  # Увеличиваем суммарное расстояние
        route.append(f"{last_node} -> {next_node} = {min_distance}")  # Добавляем шаг в маршрут

    # Возвращаемся в начальный узел
    for edge in edges:
        if (edge[0] == visited[-1] and edge[1] == start_node) or (edge[1] == visited[-1] and edge[0] == start_node):
            total_distance += edge[2]  # Увеличиваем суммарное расстояние
            route.append(f"{visited[-1]} -> {start_node} = {edge[2]}")  # Добавляем шаг в маршрут
            break

    return route, total_distance

# Функция для поиска кратчайшего пути (Алгоритм Форда-Беллмана)
def ford_bellman_algorithm():
    # Проверяем, есть ли узлы и маршруты
    if not nodes or not edges:
        return None, 0, []

    start_node = list(nodes.keys())[0]  # Начинаем с первого узла
    distance = {node: float('inf') for node in nodes}  # Инициализируем расстояния до узлов
    distance[start_node] = 0  # Расстояние до начального узла равно 0

    # Релаксация ребер
    for _ in range(len(nodes) - 1):
        for edge in edges:
            if distance[edge[0]] + edge[2] < distance[edge[1]]:
                distance[edge[1]] = distance[edge[0]] + edge[2]  # Обновляем расстояние
            if distance[edge[1]] + edge[2] < distance[edge[0]]:
                distance[edge[0]] = distance[edge[1]] + edge[2]  # Обновляем расстояние

    # Построение маршрута
    visited = [start_node]  # Список посещенных узлов
    total_distance = 0  # Суммарное расстояние
    route = []  # Маршрут

    # Пока не посетили все узлы
    while len(visited) < len(nodes):
        last_node = visited[-1]  # Последний посещенный узел
        min_distance = float('inf')  # Минимальное расстояние
        next_node = None  # Следующий узел

        # Ищем ближайший непосещенный узел
        for edge in edges:
            if edge[0] == last_node and edge[1] not in visited:
                if edge[2] < min_distance:
                    min_distance = edge[2]
                    next_node = edge[1]
            elif edge[1] == last_node and edge[0] not in visited:
                if edge[2] < min_distance:
                    min_distance = edge[2]
                    next_node = edge[0]

        # Если не нашли доступный узел, прерываем цикл
        if next_node is None:
            break

        # Добавляем узел в список посещенных
        visited.append(next_node)
        total_distance += min_distance  # Увеличиваем суммарное расстояние
        route.append(f"{last_node} -> {next_node} = {min_distance}")  # Добавляем шаг в маршрут

    # Возвращаемся в начальный узел
    for edge in edges:
        if (edge[0] == visited[-1] and edge[1] == start_node) or (edge[1] == visited[-1] and edge[0] == start_node):
            total_distance += edge[2]  # Увеличиваем суммарное расстояние
            route.append(f"{visited[-1]} -> {start_node} = {edge[2]}")  # Добавляем шаг в маршрут
            break

    return route, total_distance

# Функция для выполнения выбранного алгоритма
def run_algorithm():
    # Проверяем, есть ли узлы и маршруты
    if not nodes or not edges:
        messagebox.showerror("Ошибка", "Добавьте узлы и маршруты.")
        return

    algorithm = algorithm_var.get()  # Получаем выбранный алгоритм
    start_time = time.time()  # Начало измерения времени

    # Выполняем выбранный алгоритм
    if algorithm == "Greedy Algorithm":
        route, total_distance = greedy_algorithm()
    elif algorithm == "Ford-Bellman":
        route, total_distance = ford_bellman_algorithm()

    end_time = time.time()  # Конец измерения времени
    execution_time = (end_time - start_time) * 1000  # Время в миллисекундах

    # Если маршрут не найден, выводим ошибку
    if not route:
        messagebox.showerror("Ошибка", "Не удалось найти маршрут.")
        return

    # Вывод результатов
    result_window = tk.Toplevel()
    result_window.title("Результаты выполнения алгоритма")

    # Выводим метрики
    tk.Label(result_window, text=f"Алгоритм: {algorithm}").pack(padx=10, pady=5)
    tk.Label(result_window, text=f"Время выполнения: {execution_time:.2f} мс").pack(padx=10, pady=5)
    tk.Label(result_window, text=f"Количество посещенных узлов: {len(nodes)}").pack(padx=10, pady=5)
    tk.Label(result_window, text=f"Суммарное расстояние: {total_distance}").pack(padx=10, pady=5)

    # Выводим маршрут
    tk.Label(result_window, text="Маршрут:").pack(padx=10, pady=5)
    for step in route:
        tk.Label(result_window, text=step).pack(padx=10, pady=2)

# Функция для добавления нового узла
def add_node():
    # Запрашиваем имя узла
    node_name = simpledialog.askstring("Добавить узел", "Введите имя узла:")
    if node_name:
        # Проверяем, существует ли узел с таким именем
        if node_name in nodes:
            messagebox.showerror("Ошибка", "Узел с таким именем уже существует.")
        else:
            # Размещаем узел на холсте
            angle = 2 * math.pi / (len(nodes) + 1)
            x = 300 + 200 * math.cos(len(nodes) * angle)
            y = 300 + 200 * math.sin(len(nodes) * angle)
            nodes[node_name] = (x, y)
            draw_graph(canvas)  # Перерисовываем граф

# Функция для добавления нового маршрута
def add_edge():
    # Проверяем, есть ли хотя бы два узла
    if len(nodes) < 2:
        messagebox.showerror("Ошибка", "Должно быть至少 два узла для создания маршрута.")
        return

    # Создаем окно для добавления маршрута
    edge_window = tk.Toplevel()
    edge_window.title("Добавить маршрут")

    # Поле для выбора начального узла
    tk.Label(edge_window, text="Начальный узел:").grid(row=0, column=0, padx=10, pady=5)
    start_node_var = tk.StringVar(value=list(nodes.keys())[0])
    start_node_menu = tk.OptionMenu(edge_window, start_node_var, *nodes.keys())
    start_node_menu.grid(row=0, column=1, padx=10, pady=5)

    # Поле для выбора конечного узла
    tk.Label(edge_window, text="Конечный узел:").grid(row=1, column=0, padx=10, pady=5)
    end_node_var = tk.StringVar(value=list(nodes.keys())[1])
    end_node_menu = tk.OptionMenu(edge_window, end_node_var, *nodes.keys())
    end_node_menu.grid(row=1, column=1, padx=10, pady=5)

    # Поле для ввода расстояния
    tk.Label(edge_window, text="Расстояние:").grid(row=2, column=0, padx=10, pady=5)
    distance_entry = tk.Entry(edge_window)
    distance_entry.grid(row=2, column=1, padx=10, pady=5)

    # Кнопка для подтверждения
    def confirm_edge():
        start_node = start_node_var.get()
        end_node = end_node_var.get()
        distance = distance_entry.get()

        # Проверяем, что начальный и конечный узлы не совпадают
        if start_node == end_node:
            messagebox.showerror("Ошибка", "Начальный и конечный узлы не могут совпадать.")
            return

        # Проверяем, что расстояние является положительным числом
        try:
            distance = int(distance)
            if distance <= 0:
                messagebox.showerror("Ошибка", "Расстояние должно быть положительным числом.")
                return
        except ValueError:
            messagebox.showerror("Ошибка", "Расстояние должно быть числом.")
            return

        # Добавляем маршрут
        edges.append((start_node, end_node, distance))
        draw_graph(canvas)  # Перерисовываем граф
        edge_window.destroy()  # Закрываем окно

    tk.Button(edge_window, text="Добавить", command=confirm_edge).grid(row=3, column=0, columnspan=2, pady=10)

# Создаем главное окно приложения
root = tk.Tk()
root.title("Почтовый маршрут")

# Создаем и размещаем метку для выбора алгоритма
label_algorithm = tk.Label(root, text="Выберите алгоритм:")
label_algorithm.pack(padx=10, pady=5)

# Создаем переменную для хранения выбранного алгоритма
algorithm_var = tk.StringVar(value="Greedy Algorithm")

# Создаем радиокнопки для выбора алгоритма
greedy_radio = tk.Radiobutton(root, text="Алгоритм первый-лучший", variable=algorithm_var, value="Greedy Algorithm")
greedy_radio.pack(padx=10, pady=5)

ford_bellman_radio = tk.Radiobutton(root, text="Алгоритм Форда-Беллмана", variable=algorithm_var, value="Ford-Bellman")
ford_bellman_radio.pack(padx=10, pady=5)

# Кнопка для добавления узла
add_node_button = tk.Button(root, text="Добавить узел", command=add_node)
add_node_button.pack(padx=10, pady=5)

# Кнопка для добавления маршрута
add_edge_button = tk.Button(root, text="Добавить маршрут", command=add_edge)
add_edge_button.pack(padx=10, pady=5)

# Создаем кнопку "Выполнить" и связываем ее с функцией run_algorithm
submit_button = tk.Button(root, text="Выполнить", command=run_algorithm)
submit_button.pack(padx=10, pady=20)

# Создаем холст для визуализации графа
canvas = tk.Canvas(root, width=600, height=600)
canvas.pack(padx=10, pady=10)

# Запускаем главный цикл обработки событий
root.mainloop() 


: 

In [1]:
print('1234')

1234


In [1]:
import tkinter as tk
from tkinter import messagebox, simpledialog
import math

# --- ЛОГИКА ГРАФА ---

class MailGraph:
    def __init__(self):
        self.nodes = {}  # id -> {x, y, prio, name}
        self.edges = []  # (id1, id2, weight)
        self.node_count = 0

    def add_node(self, x, y):
        self.node_count += 1
        name = f"Отделение {self.node_count}"
        if self.node_count == 1: name = "ПОЧТАМТ"
        # По умолчанию приоритет 5, для Почтамта - 0 (высший)
        prio = 0 if self.node_count == 1 else 5
        self.nodes[self.node_count] = {'x': x, 'y': y, 'prio': prio, 'name': name}
        return self.node_count

    def add_edge(self, n1, n2, weight):
        # Проверяем, нет ли уже такой дороги
        for u, v, w in self.edges:
            if (u == n1 and v == n2) or (u == n2 and v == n1):
                return
        self.edges.append((n1, n2, weight))

# --- АЛГОРИТМЫ ---

# 1. Алгоритм Форда-Беллмана (Кратчайший путь между А и Б)
def bellman_ford(graph, start_id, end_id):
    if not start_id or not end_id: return None, 0
    
    dist = {n: float('inf') for n in graph.nodes}
    parent = {n: None for n in graph.nodes}
    dist[start_id] = 0

    for _ in range(len(graph.nodes) - 1):
        for u, v, w in graph.edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                parent[v] = u
            if dist[v] + w < dist[u]:
                dist[u] = dist[v] + w
                parent[u] = v

    path = []
    curr = end_id
    if dist[end_id] == float('inf'): return None, 0
    while curr is not None:
        path.append(curr)
        curr = parent[curr]
    return path[::-1], dist[end_id]

# 2. Жадный алгоритм (Первый-лучший) для объезда всех
def greedy_tsp(graph):
    if not graph.nodes: return None, 0
    start_node = 1 # Главпочтамт
    unvisited = [n for n in graph.nodes if n != start_node]
    path = [start_node]
    total_dist = 0
    curr = start_node

    while unvisited:
        next_node = None
        min_w = float('inf')
        for u, v, w in graph.edges:
            node_to_check = None
            if u == curr and v in unvisited: node_to_check = v
            elif v == curr and u in unvisited: node_to_check = u
            
            if node_to_check and w < min_w:
                min_w = w
                next_node = node_to_check
        
        if next_node is None: break # Тупик
        path.append(next_node)
        unvisited.remove(next_node)
        total_dist += min_w
        curr = next_node

    # Возврат
    for u, v, w in graph.edges:
        if (u == curr and v == start_node) or (v == curr and u == start_node):
            total_dist += w
            path.append(start_node)
            break
    return path, total_dist

# --- ИНТЕРФЕЙС ---

class App:
    def __init__(self, root):
        self.root = root
        self.root.title("Почтовая Логистика")
        self.graph = MailGraph()
        self.selected_node = None
        self.mode = "node" # node или edge

        # Верхняя панель управления
        control_panel = tk.Frame(root, bg="#eee", pady=5)
        control_panel.pack(side=tk.TOP, fill=tk.X)

        tk.Label(control_panel, text="Алгоритм:", bg="#eee").pack(side=tk.LEFT, padx=5)
        self.algo_var = tk.StringVar(value="greedy")
        tk.Radiobutton(control_panel, text="Жадный (TSP)", variable=self.algo_var, value="greedy", bg="#eee").pack(side=tk.LEFT)
        tk.Radiobutton(control_panel, text="Форд-Беллман (Путь)", variable=self.algo_var, value="ford", bg="#eee").pack(side=tk.LEFT)

        self.btn_node = tk.Button(control_panel, text="➕ Отделения", command=lambda: self.set_mode("node"), bg="lightblue")
        self.btn_node.pack(side=tk.LEFT, padx=10)
        self.btn_edge = tk.Button(control_panel, text="🔗 Дороги", command=lambda: self.set_mode("edge"))
        self.btn_edge.pack(side=tk.LEFT)

        tk.Button(control_panel, text="🚀 РАССЧИТАТЬ", command=self.calculate, bg="#4CAF50", fg="white").pack(side=tk.RIGHT, padx=10)
        tk.Button(control_panel, text="🗑 Очистить", command=self.clear_all).pack(side=tk.RIGHT)

        # Холст
        self.canvas = tk.Canvas(root, width=800, height=600, bg="white")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        self.canvas.bind("<Button-1>", self.on_click)

        # Подсказка снизу
        self.status = tk.Label(root, text="Кликните на поле, чтобы добавить Главпочтамт", bd=1, relief=tk.SUNKEN, anchor=tk.W)
        self.status.pack(side=tk.BOTTOM, fill=tk.X)

    def set_mode(self, mode):
        self.mode = mode
        self.selected_node = None
        self.btn_node.config(bg="lightblue" if mode == "node" else "systembuttonface")
        self.btn_edge.config(bg="lightblue" if mode == "edge" else "systembuttonface")
        self.status.config(text="Режим: Добавление отделений" if mode == "node" else "Режим: Соединение отделений дорогами")

    def on_click(self, event):
        if self.mode == "node":
            node_id = self.graph.add_node(event.x, event.y)
            self.draw()
        elif self.mode == "edge":
            clicked = self.find_node(event.x, event.y)
            if clicked:
                if self.selected_node is None:
                    self.selected_node = clicked
                    self.status.config(text=f"Выбрано: {self.graph.nodes[clicked]['name']}. Теперь выберите второе.")
                else:
                    dist = simpledialog.askinteger("Дорога", "Введите расстояние:", initialvalue=10)
                    if dist:
                        self.graph.add_edge(self.selected_node, clicked, dist)
                    self.selected_node = None
                    self.draw()

    def find_node(self, x, y):
        for nid, data in self.graph.nodes.items():
            if math.hypot(data['x'] - x, data['y'] - y) < 25:
                return nid
        return None

    def draw(self):
        self.canvas.delete("all")
        # Рисуем ребра
        for u, v, w in self.graph.edges:
            x1, y1 = self.graph.nodes[u]['x'], self.graph.nodes[u]['y']
            x2, y2 = self.graph.nodes[v]['x'], self.graph.nodes[v]['y']
            self.canvas.create_line(x1, y1, x2, y2, width=2, fill="gray")
            self.canvas.create_text((x1+x2)/2, (y1+y2)/2, text=str(w), fill="blue", font=("Arial", 10, "bold"))

        # Рисуем узлы
        for nid, data in self.graph.nodes.items():
            color = "orange" if nid == 1 else "white"
            if nid == self.selected_node: color = "yellow"
            self.canvas.create_oval(data['x']-20, data['y']-20, data['x']+20, data['y']+20, fill=color, width=2)
            self.canvas.create_text(data['x'], data['y']-30, text=data['name'], font=("Arial", 8, "bold"))
            self.canvas.create_text(data['x'], data['y'], text=f"P:{data['prio']}")

    def calculate(self):
        if not self.graph.nodes: return
        algo = self.algo_var.get()
        
        if algo == "greedy":
            # Маршрут по всем точкам
            path, dist = greedy_tsp(self.graph)
            if path:
                names = [self.graph.nodes[n]['name'] for n in path]
                messagebox.showinfo("Результат (Жадный)", f"Полный объезд почты:\n{' -> '.join(names)}\n\nДистанция: {dist}")
        
        elif algo == "ford":
            # Путь через Почтамт между А и Б
            a = simpledialog.askinteger("Путь", "ID начального отделения:")
            b = simpledialog.askinteger("Путь", "ID конечного отделения:")
            if a in self.graph.nodes and b in self.graph.nodes:
                p1, d1 = bellman_ford(self.graph, a, 1)
                p2, d2 = bellman_ford(self.graph, 1, b)
                if p1 and p2:
                    full_p = p1 + p2[1:]
                    names = [self.graph.nodes[n]['name'] for n in full_p]
                    messagebox.showinfo("Результат (Форд-Беллман)", f"Путь через Почтамт:\n{' -> '.join(names)}\n\nДистанция: {d1+d2}")
                else:
                    messagebox.showerror("Ошибка", "Путь не найден!")

    def clear_all(self):
        self.graph = MailGraph()
        self.draw()
        self.status.config(text="Граф очищен")

if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()

2026-07-14 21:44:41.689 python[9252:33954649] +[IMKClient subclass]: chose IMKClient_Modern
2026-07-14 21:44:41.689 python[9252:33954649] +[IMKInputSession subclass]: chose IMKInputSession_Modern
Exception in Tkinter callback
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/tkinter/__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "/var/folders/49/_klt8_zj7v7cbz0z02x4jfjr0000gn/T/ipykernel_9252/841666335.py", line 110, in <lambda>
    self.btn_node = tk.Button(control_panel, text="➕ Отделения", command=lambda: self.set_mode("node"), bg="lightblue")
                                                                                  ^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/49/_klt8_zj7v7cbz0z02x4jfjr0000gn/T/ipykernel_9252/841666335.py", line 131, in set_mode
    self.btn_edge.config(bg="lightblue" if mode == "edge" else "systembuttonface")
  File "/opt/anaconda3/lib/python3.12/tkinter/__init__.py", line 1722, in c

: 

In [1]:

import tkinter as tk
from tkinter import ttk, messagebox
import math
 
INF = math.inf
 
 
# ---------------------------------------------------------------------------
# Алгоритм 1: "первый-лучший" (жадный / nearest-neighbour best-first)
# ---------------------------------------------------------------------------
def best_first_route(dist_matrix, names, start=0):
    """
    Строит маршрут почтовой машины: старт в узле start (почтамт),
    на каждом шаге едем в БЛИЖАЙШИЙ ещё не посещённый пункт (это и есть
    идея алгоритма "первый-лучший" - жадно берём локально лучший вариант),
    в конце возвращаемся в start.
 
    Возвращает (route_steps, total, order), где:
      route_steps - список строк "Имя_A -> Имя_B = вес"
      total       - суммарная длина маршрута
      order       - список индексов узлов в порядке посещения (включая
                    возврат в start в конце)
    """
    n = len(names)
    visited = [False] * n
    visited[start] = True
    order = [start]
    total = 0.0
    steps = []
    current = start
 
    for _ in range(n - 1):
        best_next = None
        best_w = INF
        for j in range(n):
            if not visited[j] and dist_matrix[current][j] < best_w:
                best_w = dist_matrix[current][j]
                best_next = j
        if best_next is None:
            # из текущего узла нет прямой дороги ни к одному непосещённому -
            # такое возможно, если граф неполный; в этом случае просто
            # прыгаем к ближайшему из оставшихся по кратчайшему пути
            # (через Форда-Беллмана), чтобы маршрут не прерывался
            dist_from_current, prev = ford_bellman(dist_matrix, current)
            best_next, best_w = None, INF
            for j in range(n):
                if not visited[j] and dist_from_current[j] < best_w:
                    best_w = dist_from_current[j]
                    best_next = j
            if best_next is None:
                break
        visited[best_next] = True
        steps.append(f"{names[current]} -> {names[best_next]} = {best_w:g}")
        total += best_w
        order.append(best_next)
        current = best_next
 
    # возврат на почтамт
    back = dist_matrix[current][start]
    if back == INF:
        dist_from_current, _ = ford_bellman(dist_matrix, current)
        back = dist_from_current[start]
    steps.append(f"{names[current]} -> {names[start]} = {back:g}")
    total += back
    order.append(start)
    return steps, total, order
 
 
# ---------------------------------------------------------------------------
# Маршрут по приоритетам отделений
# ---------------------------------------------------------------------------
def priority_route(dist_matrix, names, priority, start=0):
    """
    Маршрут строится не по близости, а по важности отделений: сначала
    посещаются отделения с более высоким приоритетом (меньшее число =
    выше приоритет), затем менее приоритетные. При равном приоритете
    выбирается более близкое отделение.
 
    Расстояние между последовательными пунктами маршрута считается через
    Форда-Беллмана - это гарантирует корректный (кратчайший) переезд,
    даже если между двумя конкретными отделениями нет прямой дороги.
    """
    n = len(names)
    order_by_priority = sorted(
        [i for i in range(n) if i != start],
        key=lambda i: (priority[i], 0)
    )
 
    steps = []
    total = 0.0
    order = [start]
    current = start
    for nxt in order_by_priority:
        dist_from_current, _ = ford_bellman(dist_matrix, current)
        w = dist_from_current[nxt]
        steps.append(f"{names[current]} -> {names[nxt]} "
                      f"(приоритет {priority[nxt]}) = {w:g}")
        total += w
        order.append(nxt)
        current = nxt
 
    dist_from_current, _ = ford_bellman(dist_matrix, current)
    back = dist_from_current[start]
    steps.append(f"{names[current]} -> {names[start]} = {back:g}")
    total += back
    order.append(start)
    return steps, total, order
 
 
# ---------------------------------------------------------------------------
# Алгоритм 2: Форд-Беллман
# ---------------------------------------------------------------------------
def ford_bellman(dist_matrix, source):
    """
    Классический алгоритм Форда-Беллмана от источника source.
    Граф неориентированный: расстояние симметрично, поэтому для каждой
    пары (i, j) с известным весом добавляем в список рёбер оба
    направления (i->j) и (j->i) и релаксируем их.
 
    Возвращает (dist, prev):
      dist[i] - кратчайшее расстояние от source до i
      prev[i] - предыдущий узел на кратчайшем пути (для восстановления
                маршрута), либо None
    """
    n = len(dist_matrix)
    edges = []
    for i in range(n):
        for j in range(n):
            if i != j and dist_matrix[i][j] < INF:
                edges.append((i, j, dist_matrix[i][j]))
 
    dist = [INF] * n
    prev = [None] * n
    dist[source] = 0.0
 
    for _ in range(n - 1):
        changed = False
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                changed = True
        if not changed:
            break
 
    # проверка на отрицательные циклы (расстояния отрицательными быть не
    # должны, но проверка оставлена для корректности алгоритма)
    for u, v, w in edges:
        if dist[u] + w < dist[v] - 1e-9:
            raise ValueError("Обнаружен цикл отрицательного веса")
 
    return dist, prev
 
 
def reconstruct_path(prev, source, target):
    """Восстанавливает список узлов пути source -> target по массиву prev."""
    if target != source and prev[target] is None:
        return None
    path = [target]
    while path[-1] != source:
        path.append(prev[path[-1]])
    path.reverse()
    return path
 
 
def shortest_path_via_hub(dist_matrix, names, hub, a, b):
    """
    Задача (3): кратчайший путь из отделения a в отделение b,
    обязательно через главпочтамт hub.
 
    Считаем один раз Форда-Беллмана от hub (расстояния до всех узлов и
    предки для восстановления пути). Так как граф неориентированный,
    путь hub -> a, развёрнутый в обратную сторону, и есть путь a -> hub.
    Итоговый путь = (a -> hub) + (hub -> b), узел hub не дублируется.
    """
    dist_from_hub, prev = ford_bellman(dist_matrix, hub)
    if dist_from_hub[a] == INF or dist_from_hub[b] == INF:
        return None, INF, []
 
    path_hub_to_a = reconstruct_path(prev, hub, a)
    path_hub_to_b = reconstruct_path(prev, hub, b)
    path_a_to_hub = list(reversed(path_hub_to_a))
 
    # Внимание: путь "туда" (A -> Почтамт) и путь "обратно" (Почтамт -> B)
    # могут частично идти по одним и тем же отделениям - это нормально:
    # по условию машина обязана заехать на почтамт, поэтому реальный путь
    # действительно может проходить через один и тот же пункт дважды.
    # Поэтому показываем это явно как два отдельных этапа, а не как один
    # маршрут без повторов.
    full_path = path_a_to_hub + path_hub_to_b[1:]
 
    total = dist_from_hub[a] + dist_from_hub[b]
    steps = [f"--- Этап 1: {names[a]} -> Почтамт ---"]
    for i in range(len(path_a_to_hub) - 1):
        u, v = path_a_to_hub[i], path_a_to_hub[i + 1]
        steps.append(f"{names[u]} -> {names[v]} = {dist_matrix[u][v]:g}")
    steps.append(f"--- Этап 2: Почтамт -> {names[b]} ---")
    for i in range(len(path_hub_to_b) - 1):
        u, v = path_hub_to_b[i], path_hub_to_b[i + 1]
        steps.append(f"{names[u]} -> {names[v]} = {dist_matrix[u][v]:g}")
    return steps, total, full_path
 
 
# ---------------------------------------------------------------------------
# Графический интерфейс
# ---------------------------------------------------------------------------
class PostRouterApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Почтовые маршруты: почтамт и отделения связи")
 
        self.n_offices = tk.IntVar(value=5)
        self.entries = []       # entries[i][j] - Entry с расстоянием
        self.priority_entries = []  # приоритет для каждого отделения (без почтамта)
        self.names = []
        self.dist_matrix = None
        self.priority = None
        self.last_order = None  # последний найденный маршрут (для схемы)
 
        self._build_setup_frame()
 
    # ---------- верхняя панель: сколько отделений -------------------------
    def _build_setup_frame(self):
        top = ttk.Frame(self.root, padding=8)
        top.pack(fill="x")
 
        ttk.Label(top, text="Количество районных отделений связи:").pack(side="left")
        spin = ttk.Spinbox(top, from_=2, to=12, width=5, textvariable=self.n_offices)
        spin.pack(side="left", padx=6)
 
        ttk.Button(top, text="Сформировать таблицы",
                   command=self._build_tables).pack(side="left", padx=6)
        ttk.Button(top, text="Заполнить тестовыми данными",
                   command=self._fill_demo_data).pack(side="left", padx=6)
 
        self.tables_frame = ttk.Frame(self.root, padding=8)
        self.tables_frame.pack(fill="both", expand=True)
 
        actions = ttk.Frame(self.root, padding=8)
        actions.pack(fill="x")
        ttk.Button(actions, text="1) Маршрут мин. длины (первый-лучший)",
                   command=self.run_best_first).pack(side="left", padx=4)
        ttk.Button(actions, text="2) Маршрут по приоритетам",
                   command=self.run_priority).pack(side="left", padx=4)
        ttk.Button(actions, text="3) Путь между отделениями через почтамт",
                   command=self.run_via_hub_dialog).pack(side="left", padx=4)
 
        out_frame = ttk.Frame(self.root, padding=8)
        out_frame.pack(fill="both", expand=True)
 
        left = ttk.Frame(out_frame)
        left.pack(side="left", fill="both", expand=True)
        ttk.Label(left, text="Результат:").pack(anchor="w")
        self.output = tk.Text(left, width=48, height=18)
        self.output.pack(fill="both", expand=True)
 
        right = ttk.Frame(out_frame)
        right.pack(side="left", fill="both", expand=True)
        ttk.Label(right, text="Схема найденного маршрута:").pack(anchor="w")
        self.canvas = tk.Canvas(right, width=420, height=420, bg="white")
        self.canvas.pack(fill="both", expand=True)
 
        self._build_tables()
 
    # ---------- таблицы ввода: расстояния + приоритеты --------------------
    def _build_tables(self):
        for w in self.tables_frame.winfo_children():
            w.destroy()
 
        n = self.n_offices.get()
        self.names = ["Почтамт"] + [f"Отд.{i}" for i in range(1, n + 1)]
        size = n + 1
 
        dist_box = ttk.LabelFrame(self.tables_frame, text="Матрица расстояний "
                                   "(0 или пусто = прямой дороги нет)")
        dist_box.pack(side="left", fill="both", expand=True, padx=4)
 
        self.entries = [[None] * size for _ in range(size)]
        # заголовки столбцов
        ttk.Label(dist_box, text="").grid(row=0, column=0)
        for j in range(size):
            ttk.Label(dist_box, text=self.names[j], width=8).grid(row=0, column=j + 1)
        for i in range(size):
            ttk.Label(dist_box, text=self.names[i], width=8).grid(row=i + 1, column=0)
            for j in range(size):
                e = ttk.Entry(dist_box, width=6)
                e.grid(row=i + 1, column=j + 1, padx=1, pady=1)
                if i == j:
                    e.insert(0, "0")
                    e.configure(state="disabled")
                self.entries[i][j] = e
 
        prio_box = ttk.LabelFrame(self.tables_frame, text="Приоритет отделений\n"
                                   "(меньше число = выше приоритет)")
        prio_box.pack(side="left", fill="y", padx=4)
        self.priority_entries = []
        for i in range(1, size):
            row = ttk.Frame(prio_box)
            row.pack(fill="x", pady=2)
            ttk.Label(row, text=self.names[i], width=8).pack(side="left")
            e = ttk.Entry(row, width=5)
            e.insert(0, "1")
            e.pack(side="left")
            self.priority_entries.append(e)
 
    def _fill_demo_data(self):
        """Быстрое заполнение таблиц демонстрационными данными для проверки."""
        n = self.n_offices.get()
        size = n + 1
        import random
        random.seed(1)
        for i in range(size):
            for j in range(size):
                if i == j:
                    continue
                if self.entries[i][j].cget("state") == "disabled":
                    continue
                w = self.entries[i][j].get()
                if w.strip() == "":
                    val = random.randint(3, 25)
                    # делаем граф связным, но не полным (примерно 65% рёбер)
                    if random.random() < 0.65 or i == 0 or j == 0:
                        self.entries[i][j].delete(0, tk.END)
                        self.entries[i][j].insert(0, str(val))
                        self.entries[j][i].delete(0, tk.END)
                        self.entries[j][i].insert(0, str(val))
        for e in self.priority_entries:
            e.delete(0, tk.END)
            e.insert(0, str(random.randint(1, 3)))
 
    # ---------- сбор данных из таблиц --------------------------------------
    def _read_matrix(self):
        size = len(self.names)
        matrix = [[INF] * size for _ in range(size)]
        for i in range(size):
            matrix[i][i] = 0.0
            for j in range(size):
                if i == j:
                    continue
                txt = self.entries[i][j].get().strip()
                if txt == "" or txt == "0":
                    continue
                try:
                    val = float(txt)
                except ValueError:
                    raise ValueError(f"Некорректное расстояние {self.names[i]}-{self.names[j]}")
                if val < 0:
                    raise ValueError("Расстояние не может быть отрицательным")
                matrix[i][j] = val
                matrix[j][i] = val
        return matrix
 
    def _read_priority(self):
        priority = [0]  # почтамт - индекс 0, приоритет не используется
        for e in self.priority_entries:
            txt = e.get().strip()
            try:
                priority.append(int(txt))
            except ValueError:
                raise ValueError("Приоритет должен быть целым числом")
        return priority
 
    def _check_connected(self, matrix):
        """Проверка связности графа через Форда-Беллмана от почтамта."""
        dist, _ = ford_bellman(matrix, 0)
        unreachable = [self.names[i] for i, d in enumerate(dist) if d == INF]
        if unreachable:
            raise ValueError("Недостижимы от почтамта: " + ", ".join(unreachable))
 
    # ---------- действия ----------------------------------------------------
    def run_best_first(self):
        try:
            matrix = self._read_matrix()
            self._check_connected(matrix)
        except ValueError as e:
            messagebox.showerror("Ошибка", str(e))
            return
        steps, total, order = best_first_route(matrix, self.names, start=0)
        self._show_result("Маршрут минимальной длины (алгоритм 'первый-лучший')",
                           steps, total)
        self.last_order = order
        self._draw_route(order)
 
    def run_priority(self):
        try:
            matrix = self._read_matrix()
            priority = self._read_priority()
            self._check_connected(matrix)
        except ValueError as e:
            messagebox.showerror("Ошибка", str(e))
            return
        steps, total, order = priority_route(matrix, self.names, priority, start=0)
        self._show_result("Маршрут по приоритетам отделений (алгоритм Форда-Беллмана)",
                           steps, total)
        self.last_order = order
        self._draw_route(order)
 
    def run_via_hub_dialog(self):
        win = tk.Toplevel(self.root)
        win.title("Путь между отделениями через почтамт")
        ttk.Label(win, text="Отделение A:").grid(row=0, column=0, padx=6, pady=6)
        ttk.Label(win, text="Отделение B:").grid(row=1, column=0, padx=6, pady=6)
 
        office_names = self.names[1:]  # без почтамта
        a_var = tk.StringVar(value=office_names[0])
        b_var = tk.StringVar(value=office_names[-1])
        ttk.OptionMenu(win, a_var, office_names[0], *office_names).grid(row=0, column=1)
        ttk.OptionMenu(win, b_var, office_names[-1], *office_names).grid(row=1, column=1)
 
        def confirm():
            try:
                matrix = self._read_matrix()
                self._check_connected(matrix)
            except ValueError as e:
                messagebox.showerror("Ошибка", str(e))
                return
            a = self.names.index(a_var.get())
            b = self.names.index(b_var.get())
            if a == b:
                messagebox.showerror("Ошибка", "Отделения A и B должны различаться.")
                return
            steps, total, path = shortest_path_via_hub(matrix, self.names, 0, a, b)
            win.destroy()
            self._show_result(f"Кратчайший путь {self.names[a]} -> Почтамт -> {self.names[b]}",
                               steps, total)
            self.last_order = path
            self._draw_route(path)
 
        ttk.Button(win, text="Найти путь", command=confirm).grid(
            row=2, column=0, columnspan=2, pady=8)
 
    # ---------- вывод -------------------------------------------------------
    def _show_result(self, title, steps, total):
        self.output.delete("1.0", tk.END)
        self.output.insert(tk.END, title + "\n" + "-" * len(title) + "\n\n")
        for s in steps:
            self.output.insert(tk.END, s + "\n")
        self.output.insert(tk.END, f"\nИтоговая длина маршрута: {total:g}\n")
 
    def _draw_route(self, order):
        """
        Рисует ТОЛЬКО те пункты и рёбра, что входят в найденный маршрут,
        с номерами шагов - без лишних деталей полного графа.
        """
        self.canvas.delete("all")
        unique_nodes = []
        for idx in order:
            if idx not in unique_nodes:
                unique_nodes.append(idx)
 
        cx, cy, r = 210, 210, 160
        positions = {}
        k = len(unique_nodes)
        for pos_i, node in enumerate(unique_nodes):
            angle = 2 * math.pi * pos_i / max(k, 1)
            x = cx + r * math.cos(angle)
            y = cy + r * math.sin(angle)
            positions[node] = (x, y)
 
        # рёбра маршрута с номером шага
        for step_i in range(len(order) - 1):
            u, v = order[step_i], order[step_i + 1]
            x1, y1 = positions[u]
            x2, y2 = positions[v]
            self.canvas.create_line(x1, y1, x2, y2, fill="#3366cc", width=2,
                                     arrow=tk.LAST)
            mx, my = (x1 + x2) / 2, (y1 + y2) / 2
            self.canvas.create_oval(mx - 9, my - 9, mx + 9, my + 9, fill="#ffe082", outline="")
            self.canvas.create_text(mx, my, text=str(step_i + 1), font=("Arial", 9, "bold"))
 
        # узлы
        for node, (x, y) in positions.items():
            is_hub = (node == 0)
            radius = 26 if is_hub else 22
            fill = "#ffd54f" if is_hub else "white"
            self.canvas.create_oval(x - radius, y - radius, x + radius, y + radius,
                                     fill=fill, outline="black", width=2)
            self.canvas.create_text(x, y, text=self.names[node], font=("Arial", 9, "bold", "white"))
 
 
def main():
    root = tk.Tk()
    app = PostRouterApp(root)
    root.mainloop()
 
 
if __name__ == "__main__":
    main()
 

2026-07-18 13:02:38.742 python[1833:28399] +[IMKClient subclass]: chose IMKClient_Modern
2026-07-18 13:02:38.742 python[1833:28399] +[IMKInputSession subclass]: chose IMKInputSession_Modern
Exception in Tkinter callback
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/tkinter/__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "/var/folders/49/_klt8_zj7v7cbz0z02x4jfjr0000gn/T/ipykernel_1833/3584704176.py", line 382, in run_best_first
    self._draw_route(order)
  File "/var/folders/49/_klt8_zj7v7cbz0z02x4jfjr0000gn/T/ipykernel_1833/3584704176.py", line 478, in _draw_route
    self.canvas.create_text(x, y, text=self.names[node], font=("Arial", 9, "bold", "white"))
  File "/opt/anaconda3/lib/python3.12/tkinter/__init__.py", line 2895, in create_text
    return self._create('text', args, kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/tkinter/__init__.py", line 2861, in _create
    

: 

In [1]:

import tkinter as tk
from tkinter import ttk, messagebox
import math
 
INF = math.inf
 
 
# ---------------------------------------------------------------------------
# Алгоритм 1: "первый-лучший" (жадный / nearest-neighbour best-first)
# ---------------------------------------------------------------------------
def best_first_route(dist_matrix, names, start=0):
    """
    Строит маршрут почтовой машины: старт в узле start (почтамт),
    на каждом шаге едем в БЛИЖАЙШИЙ ещё не посещённый пункт (это и есть
    идея алгоритма "первый-лучший" - жадно берём локально лучший вариант),
    в конце возвращаемся в start.
 
    Возвращает (route_steps, total, order), где:
      route_steps - список строк "Имя_A -> Имя_B = вес"
      total       - суммарная длина маршрута
      order       - список индексов узлов в порядке посещения (включая
                    возврат в start в конце)
    """
    n = len(names)
    visited = [False] * n
    visited[start] = True
    order = [start]
    total = 0.0
    steps = []
    current = start
 
    for _ in range(n - 1):
        best_next = None
        best_w = INF
        for j in range(n):
            if not visited[j] and dist_matrix[current][j] < best_w:
                best_w = dist_matrix[current][j]
                best_next = j
        if best_next is None:
            # из текущего узла нет прямой дороги ни к одному непосещённому -
            # такое возможно, если граф неполный; в этом случае просто
            # прыгаем к ближайшему из оставшихся по кратчайшему пути
            # (через Форда-Беллмана), чтобы маршрут не прерывался
            dist_from_current, prev = ford_bellman(dist_matrix, current)
            best_next, best_w = None, INF
            for j in range(n):
                if not visited[j] and dist_from_current[j] < best_w:
                    best_w = dist_from_current[j]
                    best_next = j
            if best_next is None:
                break
        visited[best_next] = True
        steps.append(f"{names[current]} -> {names[best_next]} = {best_w:g}")
        total += best_w
        order.append(best_next)
        current = best_next
 
    # возврат на почтамт
    back = dist_matrix[current][start]
    if back == INF:
        dist_from_current, _ = ford_bellman(dist_matrix, current)
        back = dist_from_current[start]
    steps.append(f"{names[current]} -> {names[start]} = {back:g}")
    total += back
    order.append(start)
    return steps, total, order
 
 
# ---------------------------------------------------------------------------
# Маршрут по приоритетам отделений
# ---------------------------------------------------------------------------
def priority_route(dist_matrix, names, priority, start=0):
    """
    Маршрут строится не по близости, а по важности отделений: сначала
    посещаются отделения с более высоким приоритетом (меньшее число =
    выше приоритет), затем менее приоритетные. При равном приоритете
    выбирается более близкое отделение.
 
    Расстояние между последовательными пунктами маршрута считается через
    Форда-Беллмана - это гарантирует корректный (кратчайший) переезд,
    даже если между двумя конкретными отделениями нет прямой дороги.
    """
    n = len(names)
    order_by_priority = sorted(
        [i for i in range(n) if i != start],
        key=lambda i: (priority[i], 0)
    )
 
    steps = []
    total = 0.0
    order = [start]
    current = start
    for nxt in order_by_priority:
        dist_from_current, _ = ford_bellman(dist_matrix, current)
        w = dist_from_current[nxt]
        steps.append(f"{names[current]} -> {names[nxt]} "
                      f"(приоритет {priority[nxt]}) = {w:g}")
        total += w
        order.append(nxt)
        current = nxt
 
    dist_from_current, _ = ford_bellman(dist_matrix, current)
    back = dist_from_current[start]
    steps.append(f"{names[current]} -> {names[start]} = {back:g}")
    total += back
    order.append(start)
    return steps, total, order
 
 
# ---------------------------------------------------------------------------
# Алгоритм 2: Форд-Беллман
# ---------------------------------------------------------------------------
def ford_bellman(dist_matrix, source):
    """
    Классический алгоритм Форда-Беллмана от источника source.
    Граф неориентированный: расстояние симметрично, поэтому для каждой
    пары (i, j) с известным весом добавляем в список рёбер оба
    направления (i->j) и (j->i) и релаксируем их.
 
    Возвращает (dist, prev):
      dist[i] - кратчайшее расстояние от source до i
      prev[i] - предыдущий узел на кратчайшем пути (для восстановления
                маршрута), либо None
    """
    n = len(dist_matrix)
    edges = []
    for i in range(n):
        for j in range(n):
            if i != j and dist_matrix[i][j] < INF:
                edges.append((i, j, dist_matrix[i][j]))
 
    dist = [INF] * n
    prev = [None] * n
    dist[source] = 0.0
 
    for _ in range(n - 1):
        changed = False
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                changed = True
        if not changed:
            break
 
    # проверка на отрицательные циклы (расстояния отрицательными быть не
    # должны, но проверка оставлена для корректности алгоритма)
    for u, v, w in edges:
        if dist[u] + w < dist[v] - 1e-9:
            raise ValueError("Обнаружен цикл отрицательного веса")
 
    return dist, prev
 
 
def reconstruct_path(prev, source, target):
    """Восстанавливает список узлов пути source -> target по массиву prev."""
    if target != source and prev[target] is None:
        return None
    path = [target]
    while path[-1] != source:
        path.append(prev[path[-1]])
    path.reverse()
    return path
 
 
def shortest_path_via_hub(dist_matrix, names, hub, a, b):
    """
    Задача (3): кратчайший путь из отделения a в отделение b,
    обязательно через главпочтамт hub.
 
    Считаем один раз Форда-Беллмана от hub (расстояния до всех узлов и
    предки для восстановления пути). Так как граф неориентированный,
    путь hub -> a, развёрнутый в обратную сторону, и есть путь a -> hub.
    Итоговый путь = (a -> hub) + (hub -> b), узел hub не дублируется.
    """
    dist_from_hub, prev = ford_bellman(dist_matrix, hub)
    if dist_from_hub[a] == INF or dist_from_hub[b] == INF:
        return None, INF, []
 
    path_hub_to_a = reconstruct_path(prev, hub, a)
    path_hub_to_b = reconstruct_path(prev, hub, b)
    path_a_to_hub = list(reversed(path_hub_to_a))
 
    # Внимание: путь "туда" (A -> Почтамт) и путь "обратно" (Почтамт -> B)
    # могут частично идти по одним и тем же отделениям - это нормально:
    # по условию машина обязана заехать на почтамт, поэтому реальный путь
    # действительно может проходить через один и тот же пункт дважды.
    # Поэтому показываем это явно как два отдельных этапа, а не как один
    # маршрут без повторов.
    full_path = path_a_to_hub + path_hub_to_b[1:]
 
    total = dist_from_hub[a] + dist_from_hub[b]
    steps = [f"--- Этап 1: {names[a]} -> Почтамт ---"]
    for i in range(len(path_a_to_hub) - 1):
        u, v = path_a_to_hub[i], path_a_to_hub[i + 1]
        steps.append(f"{names[u]} -> {names[v]} = {dist_matrix[u][v]:g}")
    steps.append(f"--- Этап 2: Почтамт -> {names[b]} ---")
    for i in range(len(path_hub_to_b) - 1):
        u, v = path_hub_to_b[i], path_hub_to_b[i + 1]
        steps.append(f"{names[u]} -> {names[v]} = {dist_matrix[u][v]:g}")
    return steps, total, full_path
 
 
# ---------------------------------------------------------------------------
# Графический интерфейс
# ---------------------------------------------------------------------------
class PostRouterApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Почтовые маршруты: почтамт и отделения связи")
 
        self.n_offices = tk.IntVar(value=5)
        self.entries = []       # entries[i][j] - Entry с расстоянием
        self.priority_entries = []  # приоритет для каждого отделения (без почтамта)
        self.names = []
        self.dist_matrix = None
        self.priority = None
        self.last_order = None  # последний найденный маршрут (для схемы)
 
        self._build_setup_frame()
 
    # ---------- верхняя панель: сколько отделений -------------------------
    def _build_setup_frame(self):
        top = ttk.Frame(self.root, padding=8)
        top.pack(fill="x")
 
        ttk.Label(top, text="Количество районных отделений связи:").pack(side="left")
        spin = ttk.Spinbox(top, from_=2, to=12, width=5, textvariable=self.n_offices)
        spin.pack(side="left", padx=6)
 
        ttk.Button(top, text="Сформировать таблицы",
                   command=self._build_tables).pack(side="left", padx=6)
        ttk.Button(top, text="Заполнить тестовыми данными",
                   command=self._fill_demo_data).pack(side="left", padx=6)
 
        self.tables_frame = ttk.Frame(self.root, padding=8)
        self.tables_frame.pack(fill="both", expand=True)
 
        actions = ttk.Frame(self.root, padding=8)
        actions.pack(fill="x")
        ttk.Button(actions, text="1) Маршрут мин. длины (первый-лучший)",
                   command=self.run_best_first).pack(side="left", padx=4)
        ttk.Button(actions, text="2) Маршрут по приоритетам",
                   command=self.run_priority).pack(side="left", padx=4)
        ttk.Button(actions, text="3) Путь между отделениями через почтамт",
                   command=self.run_via_hub_dialog).pack(side="left", padx=4)
 
        out_frame = ttk.Frame(self.root, padding=8)
        out_frame.pack(fill="both", expand=True)
 
        left = ttk.Frame(out_frame)
        left.pack(side="left", fill="both", expand=True)
        ttk.Label(left, text="Результат:").pack(anchor="w")
        self.output = tk.Text(left, width=48, height=18)
        self.output.pack(fill="both", expand=True)
 
        right = ttk.Frame(out_frame)
        right.pack(side="left", fill="both", expand=True)
        ttk.Label(right, text="Схема найденного маршрута:").pack(anchor="w")
        self.canvas = tk.Canvas(right, width=420, height=420, bg="white")
        self.canvas.pack(fill="both", expand=True)
 
        self._build_tables()
 
    # ---------- таблицы ввода: расстояния + приоритеты --------------------
    def _build_tables(self):
        for w in self.tables_frame.winfo_children():
            w.destroy()
 
        n = self.n_offices.get()
        self.names = ["Почтамт"] + [f"Отд.{i}" for i in range(1, n + 1)]
        size = n + 1
 
        dist_box = ttk.LabelFrame(self.tables_frame, text="Матрица расстояний "
                                   "(0 или пусто = прямой дороги нет)")
        dist_box.pack(side="left", fill="both", expand=True, padx=4)
 
        self.entries = [[None] * size for _ in range(size)]
        # заголовки столбцов
        ttk.Label(dist_box, text="").grid(row=0, column=0)
        for j in range(size):
            ttk.Label(dist_box, text=self.names[j], width=8).grid(row=0, column=j + 1)
        for i in range(size):
            ttk.Label(dist_box, text=self.names[i], width=8).grid(row=i + 1, column=0)
            for j in range(size):
                e = ttk.Entry(dist_box, width=6)
                e.grid(row=i + 1, column=j + 1, padx=1, pady=1)
                if i == j:
                    e.insert(0, "0")
                    e.configure(state="disabled")
                self.entries[i][j] = e
 
        prio_box = ttk.LabelFrame(self.tables_frame, text="Приоритет отделений\n"
                                   "(меньше число = выше приоритет)")
        prio_box.pack(side="left", fill="y", padx=4)
        self.priority_entries = []
        for i in range(1, size):
            row = ttk.Frame(prio_box)
            row.pack(fill="x", pady=2)
            ttk.Label(row, text=self.names[i], width=8).pack(side="left")
            e = ttk.Entry(row, width=5)
            e.insert(0, "1")
            e.pack(side="left")
            self.priority_entries.append(e)
 
    def _fill_demo_data(self):
        """Быстрое заполнение таблиц демонстрационными данными для проверки."""
        n = self.n_offices.get()
        size = n + 1
        import random
        random.seed(1)
        for i in range(size):
            for j in range(size):
                if i == j:
                    continue
                if self.entries[i][j].cget("state") == "disabled":
                    continue
                w = self.entries[i][j].get()
                if w.strip() == "":
                    val = random.randint(3, 25)
                    # делаем граф связным, но не полным (примерно 65% рёбер)
                    if random.random() < 0.65 or i == 0 or j == 0:
                        self.entries[i][j].delete(0, tk.END)
                        self.entries[i][j].insert(0, str(val))
                        self.entries[j][i].delete(0, tk.END)
                        self.entries[j][i].insert(0, str(val))
        for e in self.priority_entries:
            e.delete(0, tk.END)
            e.insert(0, str(random.randint(1, 3)))
 
    # ---------- сбор данных из таблиц --------------------------------------
    def _read_matrix(self):
        size = len(self.names)
        matrix = [[INF] * size for _ in range(size)]
        for i in range(size):
            matrix[i][i] = 0.0
            for j in range(size):
                if i == j:
                    continue
                txt = self.entries[i][j].get().strip()
                if txt == "" or txt == "0":
                    continue
                try:
                    val = float(txt)
                except ValueError:
                    raise ValueError(f"Некорректное расстояние {self.names[i]}-{self.names[j]}")
                if val < 0:
                    raise ValueError("Расстояние не может быть отрицательным")
                matrix[i][j] = val
                matrix[j][i] = val
        return matrix
 
    def _read_priority(self):
        priority = [0]  # почтамт - индекс 0, приоритет не используется
        for e in self.priority_entries:
            txt = e.get().strip()
            try:
                priority.append(int(txt))
            except ValueError:
                raise ValueError("Приоритет должен быть целым числом")
        return priority
 
    def _check_connected(self, matrix):
        """Проверка связности графа через Форда-Беллмана от почтамта."""
        dist, _ = ford_bellman(matrix, 0)
        unreachable = [self.names[i] for i, d in enumerate(dist) if d == INF]
        if unreachable:
            raise ValueError("Недостижимы от почтамта: " + ", ".join(unreachable))
 
    # ---------- действия ----------------------------------------------------
    def run_best_first(self):
        try:
            matrix = self._read_matrix()
            self._check_connected(matrix)
        except ValueError as e:
            messagebox.showerror("Ошибка", str(e))
            return
        steps, total, order = best_first_route(matrix, self.names, start=0)
        self._show_result("Маршрут минимальной длины (алгоритм 'первый-лучший')",
                           steps, total)
        self.last_order = order
        self._draw_route(order)
 
    def run_priority(self):
        try:
            matrix = self._read_matrix()
            priority = self._read_priority()
            self._check_connected(matrix)
        except ValueError as e:
            messagebox.showerror("Ошибка", str(e))
            return
        steps, total, order = priority_route(matrix, self.names, priority, start=0)
        self._show_result("Маршрут по приоритетам отделений (алгоритм Форда-Беллмана)",
                           steps, total)
        self.last_order = order
        self._draw_route(order)
 
    def run_via_hub_dialog(self):
        win = tk.Toplevel(self.root)
        win.title("Путь между отделениями через почтамт")
        ttk.Label(win, text="Отделение A:").grid(row=0, column=0, padx=6, pady=6)
        ttk.Label(win, text="Отделение B:").grid(row=1, column=0, padx=6, pady=6)
 
        office_names = self.names[1:]  # без почтамта
        a_var = tk.StringVar(value=office_names[0])
        b_var = tk.StringVar(value=office_names[-1])
        ttk.OptionMenu(win, a_var, office_names[0], *office_names).grid(row=0, column=1)
        ttk.OptionMenu(win, b_var, office_names[-1], *office_names).grid(row=1, column=1)
 
        def confirm():
            try:
                matrix = self._read_matrix()
                self._check_connected(matrix)
            except ValueError as e:
                messagebox.showerror("Ошибка", str(e))
                return
            a = self.names.index(a_var.get())
            b = self.names.index(b_var.get())
            if a == b:
                messagebox.showerror("Ошибка", "Отделения A и B должны различаться.")
                return
            steps, total, path = shortest_path_via_hub(matrix, self.names, 0, a, b)
            win.destroy()
            self._show_result(f"Кратчайший путь {self.names[a]} -> Почтамт -> {self.names[b]}",
                               steps, total)
            self.last_order = path
            self._draw_route(path)
 
        ttk.Button(win, text="Найти путь", command=confirm).grid(
            row=2, column=0, columnspan=2, pady=8)
 
    # ---------- вывод -------------------------------------------------------
    def _show_result(self, title, steps, total):
        self.output.delete("1.0", tk.END)
        self.output.insert(tk.END, title + "\n" + "-" * len(title) + "\n\n")
        for s in steps:
            self.output.insert(tk.END, s + "\n")
        self.output.insert(tk.END, f"\nИтоговая длина маршрута: {total:g}\n")
 
    def _draw_route(self, order):
        """
        Рисует ТОЛЬКО те пункты и рёбра, что входят в найденный маршрут,
        с номерами шагов - без лишних деталей полного графа.
        """
        self.canvas.delete("all")
        unique_nodes = []
        for idx in order:
            if idx not in unique_nodes:
                unique_nodes.append(idx)
 
        cx, cy, r = 210, 210, 160
        positions = {}
        k = len(unique_nodes)
        for pos_i, node in enumerate(unique_nodes):
            angle = 2 * math.pi * pos_i / max(k, 1)
            x = cx + r * math.cos(angle)
            y = cy + r * math.sin(angle)
            positions[node] = (x, y)
 
        # рёбра маршрута с номером шага
        for step_i in range(len(order) - 1):
            u, v = order[step_i], order[step_i + 1]
            x1, y1 = positions[u]
            x2, y2 = positions[v]
            self.canvas.create_line(x1, y1, x2, y2, fill="#3366cc", width=2,
                                     arrow=tk.LAST)
            mx, my = (x1 + x2) / 2, (y1 + y2) / 2
            self.canvas.create_oval(mx - 9, my - 9, mx + 9, my + 9, fill="#ffe082", outline="")
            self.canvas.create_text(mx, my, text=str(step_i + 1), font=("Arial", 9, "bold"))
 
        # узлы
        for node, (x, y) in positions.items():
            is_hub = (node == 0)
            radius = 26 if is_hub else 22
            fill = "#ffd54f" if is_hub else "white"
            self.canvas.create_oval(x - radius, y - radius, x + radius, y + radius,
                                     fill=fill, outline="black", width=2)
            self.canvas.create_text(x, y, text=self.names[node], font=("Arial", 9, "bold"))
 
 
def main():
    root = tk.Tk()
    app = PostRouterApp(root)
    root.mainloop()
 
 
if __name__ == "__main__":
    main()

2026-07-14 22:10:35.285 python[9421:33971706] +[IMKClient subclass]: chose IMKClient_Modern
2026-07-14 22:10:35.285 python[9421:33971706] +[IMKInputSession subclass]: chose IMKInputSession_Modern


: 

In [1]:
# -*- coding: utf-8 -*-
"""
Задача о почтовых маршрутах: почтамт и районные отделения связи.
Три подзадачи:
маршрут машины от почтамта по всем отделениям и обратно на почтамт
маршрут машины по всем отделениям, построенный по приоритетам;
кратчайший путь между двумя заданными отделениями через почтамт.
Алгоритмы поиска (как требуется в задании):
"первый-лучший" (жадный best-first) - задача (1);
Форд-Беллман - задача (2) и задача (3).
"""
import tkinter as tk
from tkinter import ttk, messagebox
import math
INF = math.inf
"""---------------------------------------------------------------------------
Алгоритм 1: "первый-лучший"
---------------------------------------------------------------------------"""
def best_first_route(dist_matrix, names, start=0):
    n = len(names)
    visited = [False] * n
    visited[start] = True
    order = [start]
    total = 0.0
    steps = []
    current = start
    for _ in range(n - 1):
        best_next, best_w = None, INF
        for j in range(n):
            if not visited[j] and dist_matrix[current][j] < best_w:
                best_w = dist_matrix[current][j]
                best_next = j
        if best_next is None:
            dist_from_current, _ = ford_bellman(dist_matrix, current)
            best_next, best_w = None, INF
            for j in range(n):
                if not visited[j] and dist_from_current[j] < best_w:
                    best_w = dist_from_current[j]
                    best_next = j
            if best_next is None:
                break
        visited[best_next] = True
        steps.append(f"{names[current]} -> {names[best_next]} = {best_w:g}")
        total += best_w
        order.append(best_next)
        current = best_next
    back = dist_matrix[current][start]
    if back == INF:
        dist_from_current, _ = ford_bellman(dist_matrix, current)
        back = dist_from_current[start]
    steps.append(f"{names[current]} -> {names[start]} = {back:g}")
    total += back
    order.append(start)
    return steps, total, order
"""---------------------------------------------------------------------------
Маршрут по приоритетам (алгоритм Форда-Беллмана используется для
получения кратчайших расстояний между отделениями на каждом шаге)
---------------------------------------------------------------------------"""
def priority_route(dist_matrix, names, priority, start=0):
    """
    На каждом шаге из ещё не посещённых пунктов выбирается тот, у кого
    приоритет ВАЖНЕЕ (меньшее число). При РАВНОМ приоритете у нескольких
    отделений выбор идёт по кратчайшему расстоянию до них - то есть, если
    ВСЕМ отделениям задать одинаковый приоритет, маршрут полностью
    сведётся к выбору ближайшего пункта на каждом шаге (будет совпадать
    с алгоритмом "первый-лучший"). Приоритет - любое целое число, лимитов
    нет; отрицательные и нулевые значения не ошибка, а лишь означают ещё
    более высокий приоритет, чем 1.
    """
    n = len(names)
    remaining = [i for i in range(n) if i != start]
    steps = []
    total = 0.0
    order = [start]
    current = start
    while remaining:
        dist_from_current, _ = ford_bellman(dist_matrix, current)
        best = min(remaining, key=lambda i: (priority[i], dist_from_current[i]))
        w = dist_from_current[best]
        steps.append(f"{names[current]} -> {names[best]} "
                     f"(приоритет {priority[best]}) = {w:g}")
        total += w
        order.append(best)
        remaining.remove(best)
        current = best
    dist_from_current, _ = ford_bellman(dist_matrix, current)
    back = dist_from_current[start]
    steps.append(f"{names[current]} -> {names[start]} = {back:g}")
    total += back
    order.append(start)
    return steps, total, order
'''---------------------------------------------------------------------------
Алгоритм 2: Форд-Беллман
---------------------------------------------------------------------------'''
def ford_bellman(dist_matrix, source):
    n = len(dist_matrix)
    edges = []
    for i in range(n):
        for j in range(n):
            if i != j and dist_matrix[i][j] < INF:
                edges.append((i, j, dist_matrix[i][j]))
    dist = [INF] * n
    prev = [None] * n
    dist[source] = 0.0
    for _ in range(n - 1):
        changed = False
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                changed = True
        if not changed:
            break
    for u, v, w in edges:
        if dist[u] + w < dist[v] - 1e-9:
            raise ValueError("Обнаружен цикл отрицательного веса")
    return dist, prev

def reconstruct_path(prev, source, target):
    if target != source and prev[target] is None:
        return None
    path = [target]
    while path[-1] != source:
        path.append(prev[path[-1]])
    path.reverse()
    return path

def shortest_path_via_hub(dist_matrix, names, hub, a, b):
    """
    Кратчайший путь из a в b обязательно через hub. Путь "туда" (a -> hub)
    и путь "обратно" (hub -> b) могут частично идти по одним и тем же
    отделениям - это нормально (машина реально обязана заехать на
    почтамт), поэтому результат показывается как два отдельных этапа.
    """
    dist_from_hub, prev = ford_bellman(dist_matrix, hub)
    if dist_from_hub[a] == INF or dist_from_hub[b] == INF:
        return None, INF, []
    path_hub_to_a = reconstruct_path(prev, hub, a)
    path_hub_to_b = reconstruct_path(prev, hub, b)
    path_a_to_hub = list(reversed(path_hub_to_a))
    full_path = path_a_to_hub + path_hub_to_b[1:]
    total = dist_from_hub[a] + dist_from_hub[b]
    steps = [f"--- Этап 1: {names[a]} -> Почтамт ---"]
    for i in range(len(path_a_to_hub) - 1):
        u, v = path_a_to_hub[i], path_a_to_hub[i + 1]
        steps.append(f"{names[u]} -> {names[v]} = {dist_matrix[u][v]:g}")
    steps.append(f"--- Этап 2: Почтамт -> {names[b]} ---")
    for i in range(len(path_hub_to_b) - 1):
        u, v = path_hub_to_b[i], path_hub_to_b[i + 1]
        steps.append(f"{names[u]} -> {names[v]} = {dist_matrix[u][v]:g}")
    return steps, total, full_path
"""---------------------------------------------------------------------------
Прокручиваемый контейнер (нужен и для таблиц ввода, и для схемы
маршрута - при большом числе отделений содержимое не помещается в окно)
---------------------------------------------------------------------------"""
class ScrollableArea(ttk.Frame):
    def __init__(self, parent, width=700, height=350, bg=None):
        super().__init__(parent)
        # Если цвет фона не задан явно - берём цвет стиля TFrame,
        # чтобы холст сливался с общим фоном окна и не было белых пятен.
        if bg is None:
            style = ttk.Style(self)
            bg = style.lookup("TFrame", "background")
            if not bg:
                bg = parent.winfo_toplevel().cget("bg")
        self.canvas = tk.Canvas(self, width=width, height=height,
                                highlightthickness=0, bg=bg)
        vbar = ttk.Scrollbar(self, orient="vertical", command=self.canvas.yview)
        hbar = ttk.Scrollbar(self, orient="horizontal", command=self.canvas.xview)
        self.canvas.configure(yscrollcommand=vbar.set, xscrollcommand=hbar.set)
        self.canvas.grid(row=0, column=0, sticky="nsew")
        vbar.grid(row=0, column=1, sticky="ns")
        hbar.grid(row=1, column=0, sticky="ew")
        self.rowconfigure(0, weight=1)
        self.columnconfigure(0, weight=1)
        # внутренний Frame используется для таблиц; для схемы маршрута
        # рисование идёт прямо на self.canvas
        self.inner = ttk.Frame(self.canvas)
        self.canvas.create_window((0, 0), window=self.inner, anchor="nw")
        self.inner.bind("<Configure>",
                        lambda e: self.canvas.configure(scrollregion=self.canvas.bbox("all")))
        self.canvas.bind("<Enter>", self._bind_wheel)
        self.canvas.bind("<Leave>", self._unbind_wheel)

    def _bind_wheel(self, _e):
        self.canvas.bind_all("<MouseWheel>", self._on_wheel)
        self.canvas.bind_all("<Button-4>", self._on_wheel_linux)
        self.canvas.bind_all("<Button-5>", self._on_wheel_linux)

    def _unbind_wheel(self, _e):
        self.canvas.unbind_all("<MouseWheel>")
        self.canvas.unbind_all("<Button-4>")
        self.canvas.unbind_all("<Button-5>")

    def _on_wheel(self, event):
        self.canvas.yview_scroll(-1 if event.delta > 0 else 1, "units")

    def _on_wheel_linux(self, event):
        self.canvas.yview_scroll(-1 if event.num == 4 else 1, "units")
"""---------------------------------------------------------------------------
Графический интерфейс
---------------------------------------------------------------------------"""
class PostRouterApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Почтовые маршруты: почтамт и отделения связи")
        self.n_offices = tk.IntVar(value=5)
        self.entries = []
        self.priority_entries = []
        self.names = []
        self.last_order = None
        self._last_matrix = None
        self._build_setup_frame()

    def _build_setup_frame(self):
        top = ttk.Frame(self.root, padding=8)
        top.pack(fill="x")
        ttk.Label(top, text="Количество районных отделений связи:").pack(side="left")
        spin = ttk.Spinbox(top, from_=2, to=40, width=5, textvariable=self.n_offices)
        spin.pack(side="left", padx=6)
        ttk.Button(top, text="Сформировать таблицы",
                   command=self._build_tables).pack(side="left", padx=6)
        ttk.Button(top, text="Заполнить тестовыми данными",
                   command=self._fill_demo_data).pack(side="left", padx=6)
        # таблицы ввода - в прокручиваемой области (при 15+ отделениях
        # они физически не помещаются на экране без прокрутки)
        self.tables_area = ScrollableArea(self.root, width=920, height=280)
        self.tables_area.pack(fill="x", padx=8, pady=4)
        self.tables_frame = self.tables_area.inner
        actions = ttk.Frame(self.root, padding=8)
        actions.pack(fill="x")
        ttk.Button(actions, text="Алгоритм «первый-лучший»: маршрут мин. длины",
                   command=self.run_best_first).pack(side="left", padx=4)
        ttk.Button(actions, text="Алгоритм Форда-Беллмана: маршрут по приоритетам",
                   command=self.run_priority).pack(side="left", padx=4)
        ttk.Button(actions, text="Алгоритм Форда-Беллмана: путь через почтамт",
                   command=self.run_via_hub_dialog).pack(side="left", padx=4)
        out_frame = ttk.Frame(self.root, padding=8)
        out_frame.pack(fill="both", expand=True)
        left = ttk.Frame(out_frame)
        left.pack(side="left", fill="both", expand=True)
        ttk.Label(left, text="Результат:").pack(anchor="w")
        self.output = tk.Text(left, width=50, height=20)
        self.output.pack(fill="both", expand=True)
        right = ttk.Frame(out_frame)
        right.pack(side="left", fill="both", expand=True)
        ttk.Label(right, text="Схема маршрута (в кружке - номер отделения, "
                              "«П» - почтамт; на ребре - «шаг) вес дороги»):").pack(anchor="w")
        self.route_area = ScrollableArea(right, width=480, height=440)
        self.route_area.pack(fill="both", expand=True)
        self.canvas = self.route_area.canvas
        self._build_tables()

    # ---------- таблицы ввода: расстояния + приоритеты --------------------
    def _build_tables(self):
        for w in self.tables_frame.winfo_children():
            w.destroy()
        n = self.n_offices.get()
        self.names = ["Почтамт"] + [f"Отд.{i}" for i in range(1, n + 1)]
        size = n + 1
        dist_box = ttk.LabelFrame(self.tables_frame, text="Матрица расстояний "
                                                          "(0 или пусто = прямой дороги нет)")
        dist_box.pack(side="left", fill="both", expand=True, padx=4)
        self.entries = [[None] * size for _ in range(size)]
        ttk.Label(dist_box, text="").grid(row=0, column=0)
        for j in range(size):
            ttk.Label(dist_box, text=self.names[j], width=8).grid(row=0, column=j + 1)
        for i in range(size):
            ttk.Label(dist_box, text=self.names[i], width=8).grid(row=i + 1, column=0)
            for j in range(size):
                e = ttk.Entry(dist_box, width=6)
                e.grid(row=i + 1, column=j + 1, padx=1, pady=1)
                if i == j:
                    e.insert(0, "0")
                    e.configure(state="disabled")
                self.entries[i][j] = e
        prio_box = ttk.LabelFrame(
            self.tables_frame,
            text="Приоритет отделений\n")
        prio_box.pack(side="left", fill="y", padx=4)
        self.priority_entries = []
        for i in range(1, size):
            row = ttk.Frame(prio_box)
            row.pack(fill="x", pady=2)
            ttk.Label(row, text=self.names[i], width=8).pack(side="left")
            e = ttk.Entry(row, width=6)
            e.insert(0, "1")
            e.pack(side="left")
            self.priority_entries.append(e)

    def _fill_demo_data(self):
        n = self.n_offices.get()
        size = n + 1
        import random
        random.seed(1)
        for i in range(size):
            for j in range(size):
                if i == j:
                    continue
                if self.entries[i][j].cget("state") == "disabled":
                    continue
                if self.entries[i][j].get().strip() == "":
                    val = random.randint(3, 25)
                    if random.random() < 0.65 or i == 0 or j == 0:
                        self.entries[i][j].delete(0, tk.END)
                        self.entries[i][j].insert(0, str(val))
                        self.entries[j][i].delete(0, tk.END)
                        self.entries[j][i].insert(0, str(val))
        for e in self.priority_entries:
            e.delete(0, tk.END)
            e.insert(0, str(random.randint(1, 3)))

    # ---------- сбор данных из таблиц --------------------------------------
    def _read_matrix(self):
        size = len(self.names)
        matrix = [[INF] * size for _ in range(size)]
        for i in range(size):
            matrix[i][i] = 0.0
            for j in range(size):
                if i == j:
                    continue
                txt = self.entries[i][j].get().strip()
                if txt == "" or txt == "0":
                    continue
                try:
                    val = float(txt)
                except ValueError:
                    raise ValueError(f"Некорректное расстояние {self.names[i]}-{self.names[j]}")
                if val < 0:
                    raise ValueError("Расстояние не может быть отрицательным")
                matrix[i][j] = val
                matrix[j][i] = val
        return matrix

    def _read_priority(self):
        priority = [0]
        for e in self.priority_entries:
            txt = e.get().strip()
            try:
                priority.append(int(txt))
            except ValueError:
                raise ValueError("Приоритет должен быть целым числом (можно отрицательным)")
        return priority

    def _check_connected(self, matrix):
        dist, _ = ford_bellman(matrix, 0)
        unreachable = [self.names[i] for i, d in enumerate(dist) if d == INF]
        if unreachable:
            raise ValueError("Недостижимы от почтамта: " + ", ".join(unreachable))

    # ---------- действия ----------------------------------------------------
    def run_best_first(self):
        try:
            matrix = self._read_matrix()
            self._check_connected(matrix)
        except ValueError as e:
            messagebox.showerror("Ошибка", str(e))
            return
        steps, total, order = best_first_route(matrix, self.names, start=0)
        self._show_result("Алгоритм «первый-лучший»: маршрут минимальной длины",
                          steps, total)
        self._last_matrix = matrix
        self.last_order = order
        self._draw_route(order)

    def run_priority(self):
        try:
            matrix = self._read_matrix()
            priority = self._read_priority()
            self._check_connected(matrix)
        except ValueError as e:
            messagebox.showerror("Ошибка", str(e))
            return
        steps, total, order = priority_route(matrix, self.names, priority, start=0)
        self._show_result("Алгоритм Форда-Беллмана: маршрут по приоритетам отделений",
                          steps, total)
        self._last_matrix = matrix
        self.last_order = order
        self._draw_route(order)

    def run_via_hub_dialog(self):
        win = tk.Toplevel(self.root)
        win.title("Путь между отделениями через почтамт")
        ttk.Label(win, text="Отделение A:").grid(row=0, column=0, padx=6, pady=6)
        ttk.Label(win, text="Отделение B:").grid(row=1, column=0, padx=6, pady=6)
        office_names = self.names[1:]
        a_var = tk.StringVar(value=office_names[0])
        b_var = tk.StringVar(value=office_names[-1])
        ttk.OptionMenu(win, a_var, office_names[0], *office_names).grid(row=0, column=1)
        ttk.OptionMenu(win, b_var, office_names[-1], *office_names).grid(row=1, column=1)

        def confirm():
            try:
                matrix = self._read_matrix()
                self._check_connected(matrix)
            except ValueError as e:
                messagebox.showerror("Ошибка", str(e))
                return
            a = self.names.index(a_var.get())
            b = self.names.index(b_var.get())
            if a == b:
                messagebox.showerror("Ошибка", "Отделения A и B должны различаться.")
                return
            steps, total, path = shortest_path_via_hub(matrix, self.names, 0, a, b)
            win.destroy()
            self._show_result(
                f"Алгоритм Форда-Беллмана: путь {self.names[a]} -> Почтамт -> {self.names[b]}",
                steps, total)
            self._last_matrix = matrix
            self.last_order = path
            self._draw_route(path)

        ttk.Button(win, text="Найти путь", command=confirm).grid(
            row=2, column=0, columnspan=2, pady=8)

    # ---------- вывод -------------------------------------------------------
    def _show_result(self, title, steps, total):
        self.output.delete("1.0", tk.END)
        self.output.insert(tk.END, title + "\n" + "-" * len(title) + "\n\n")
        for s in steps:
            self.output.insert(tk.END, s + "\n")
        self.output.insert(tk.END, f"\nИтоговая длина маршрута: {total:g}\n")

    def _draw_route(self, order):
        """
        Раскладка "змейкой": узлы располагаются строго в порядке их
        посещения слева направо, с переносом на новую строку. Соседние
        по маршруту пункты почти всегда оказываются рядом на схеме,
        поэтому рёбра короткие и не пересекаются, а расстояние между
        узлами не сжимается при увеличении их числа (в отличие от
        раскладки по кругу) - вместо этого просто увеличивается схема,
        которая прокручивается скроллбарами/колесом мыши.
        """
        self.canvas.delete("all")
        unique_nodes = []
        for idx in order:
            if idx not in unique_nodes:
                unique_nodes.append(idx)
        k = len(unique_nodes)
        cols = max(3, math.ceil(math.sqrt(k)))
        col_spacing = 130
        row_spacing = 130
        margin = 70
        radius = 26
        positions = {}
        for i, node in enumerate(unique_nodes):
            row = i // cols
            col = i % cols
            if row % 2 == 1:
                col = cols - 1 - col
            x = margin + col * col_spacing
            y = margin + row * row_spacing
            positions[node] = (x, y)
        rows = math.ceil(k / cols)
        width = margin * 2 + (cols - 1) * col_spacing
        height = margin * 2 + (rows - 1) * row_spacing
        n_steps = len(order) - 1
        for step_i in range(n_steps):
            u, v = order[step_i], order[step_i + 1]
            x1, y1 = positions[u]
            x2, y2 = positions[v]
            is_last = (step_i == n_steps - 1)  # последний шаг = возврат на почтамт
            color = "#e65100" if is_last else "#3366cc"
            dash = (5, 3) if is_last else None
            self.canvas.create_line(x1, y1, x2, y2, fill=color, width=2,
                                    arrow=tk.LAST, dash=dash)
            weight = self._last_matrix[u][v] if self._last_matrix is not None else 0
            mx, my = (x1 + x2) / 2, (y1 + y2) / 2
            label = f"{step_i + 1}) {weight:g}"
            pad_w = 10 + 6 * len(label)
            self.canvas.create_rectangle(mx - pad_w / 2, my - 11, mx + pad_w / 2, my + 11,
                                         fill="#fff3cd", outline="#e6b800")
            # Явно задаём чёрный цвет текста на ребрах - так он хорошо
            # читается на светло-жёлтой подложке и на любом фоне холста.
            self.canvas.create_text(mx, my, text=label,
                                    font=("Arial", 9, "bold"), fill="black")
        # узлы: в центре круга - номер отделения (для почтамта - "П")
        for node, (x, y) in positions.items():
            is_hub = (node == 0)
            fill = "#ffd54f" if is_hub else "white"
            r = 32 if is_hub else radius
            self.canvas.create_oval(x - r, y - r, x + r, y + r,
                                    fill=fill, outline="black", width=2)
            center_text = "П" if is_hub else str(node)
            self.canvas.create_text(x, y, text=center_text, font=("Arial", 13, "bold"))
            self.canvas.create_text(x, y + r + 12, text=self.names[node],
                                    font=("Arial", 8), fill="#FFFFFF")
        self.canvas.configure(scrollregion=(0, 0, width + margin, height + margin + 20))

def main():
    root = tk.Tk()
    app = PostRouterApp(root)
    root.mainloop()

if __name__ == "__main__":
    main()

2026-07-20 19:38:48.418 python[5149:365248] +[IMKClient subclass]: chose IMKClient_Modern
2026-07-20 19:38:48.418 python[5149:365248] +[IMKInputSession subclass]: chose IMKInputSession_Modern


: 